# Exercício Prático 1: Algoritmos de Busca para Cavalos

Disciplina: Inteligência Artificial


Período: 2026.1

Nome: Pedro Costa da Motta

Matrícula: 20240014240

### 1. Definição do problema

O algoritmo vai simular o cavalo do jogo enclose horse. 
Há um espaço de estados (0 <= x <= W, 0 <=y <= H) que representa as posições possíveis do cavalo no grid, cuja coordenada (x,y) que contém o caractere 'C' é o estado inicial do problema.
Os movimentos podem ser em 4 direções (cima, baixo, esquerda e direita), e o movimento só pode ser feito se o destino (x', y') for um espaço livre ( ), cereja (J), maçã (M) ou um enxame (A). Ou seja, se o destino for uma parede (+) ou lagoa (%), o movimento não se concretiza e não é feito.
O objetivo é encostar em qualquer borda do cenário ((x = 0 || x = W-1), (y == 0 || y = H-1)), e cada movimento possui custo 1. Caso nenhuma borda consiga ser tocada, é preciso calcular a pontuação total da área que é possível se movimentar (Espaço Livre = 1; Cereja = 3; Maçã Dourada = 10; Enxame de Abelhas = -5).


### 2. Ambiente

Implementação da classe EncloseHorse.
Carregamento do arquivo de estado genérico em um grid, junto com a localização da posição inicial 'C'.
Funções para validação de movimento, verificação de fuga do cavalo e pontuação das células.

Variáveis:
  w = largura da área jogável;
  h = altura da área jogável;
  initPos = posição inicial;
  lineContent = conteúdo da linha da área;
  proccLine = linha atual que está sendo "copiada" para dentro do grid;
  dx, dy: movimentos direcionais;
  nx, ny: pontos da nova localização do cavalo no grid;


In [1]:
class EncloseHorse:
    def __init__(self, arch):
        with open(arch, 'r') as f:
            lines = f.readlines()
        self.W, self.H = map(int, lines[0].split())
        self.grid = []
        self.initPos = None
        for y, lineContent in enumerate(lines[1:self.H+1]):
            proccLine = list(lineContent.replace('\n', '').ljust(self.W))
            self.grid.append(proccLine)
            if 'C' in proccLine:
                self.initPos = (proccLine.index('C'), y)
    
    def possMovements(self, x, y):
        moves = []
        for dx, dy in [(0,1), (1,0), (0,-1), (-1,0)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.W and 0 <= ny < self.H:
                if self.grid[ny][nx] not in ['%', '+']:
                    moves.append((nx, ny))
        return moves
    
    def escape(self, x, y):
        return x == 0 or x==self.W-1 or y == 0 or y == self.H-1
    
    def cellScore(self, x, y):
        cell = self.grid[y][x]
        values = {
            ' ': 1, 'C': 1,
            'J': 3,
            'M': 10,
            'A': -5
        }
        return values.get(cell, 0)
    
    def finalScore(self, path):
        total = 0
        for x, y in path:
            total += self.cellScore(x, y)
        return total

### 3. Algoritmos de busca

Implementação das 3 funções pedidas no trabalho e a heurística admissível.
Uso do módulo: collections. 

Lógica da Busca em Largura: Os estados ficam em uma lista, no qual o mais a esquerda sempre é retirado em prioridade e então esse estado passa por uma verificação para saber se é borda ou não, e se não for, novas posições são adicionadas à fila, sendo também colocadas em um vetor com todos os estados previamente visitados (para evitar a retomada a esses estados). Essa iteração é repetida até que não haja mais estados na estrutura de dados ou que uma rota de fuga seja encontrada.

A lógica da Busca em Profundidade é praticamente igual, só com os pequenos ajustes de que a estrutura de dados é uma pilha, e a mais acima é retirada da estrutura.

Lógica da Busca A*: Os estados ficam em uma heap, no qual o estado com menor custo é retirado, com a verificação de borda. Desse estado, os vizinhos são criados (retirando os previamente visitados) e colocando o estado de menor custo na próxima posição a ser vista. Essa iteração é repetida até que não haja mais estados na estrutura de dados ou que uma rota de fuga seja encontrada.

##### Busca em largura

In [2]:
from collections import deque

def busca_largura(env):
    start = env.initPos
    if not start:
        return {
            "Resultado": "Erro",
            "Mensagem": "Cavalo não encontrado"
        }
    Queue = deque([(start, [start])])
    visited = {start}
    expNodes = 0
    while Queue:
        (x, y), path = Queue.popleft()
        expNodes += 1
        if env.escape(x, y):
            return {
                "Resultado": "Fuga",
                "Caminho": path,
                "Nos_Expandidos": expNodes,
                "Custo": len(path) - 1
            }
        for nx, ny in env.possMovements(x, y):
            if (nx, ny) not in visited:
                visited.add((nx, ny))
                nPath = path + [(nx, ny)]
                Queue.append(((nx, ny), nPath))
    return {
        "Resultado": "Cercado",
        "Visitados": visited,
        "Nos_Expandidos": expNodes
    }

##### Busca em profundidade

In [3]:
from collections import deque

def busca_profundidade(env):
    start = env.initPos
    if not start:
        return {
            "Resultado": "Erro",
            "Mensagem": "Cavalo não encontrado"
        }
    Stack = deque([(start, [start])])
    visited = {start}
    expNodes = 0
    while Stack:
        (x, y), path = Stack.pop()
        expNodes += 1
        if env.escape(x, y):
            return {
                "Resultado": "Fuga",
                "Caminho": path,
                "Nos_Expandidos": expNodes,
                "Custo": len(path) - 1
            }
        for nx, ny in env.possMovements(x, y):
            if (nx, ny) not in visited:
                visited.add((nx, ny))
                Stack.append(((nx, ny), path + [(nx, ny)]))        
    return {
        "Resultado": "Cercado",
        "Visitados": visited,
        "Nos_Expandidos": expNodes
    }

##### A*

In [4]:
import heapq

def busca_a_estrela(env):
    start = env.initPos
    if not start:
        return {"Resultado": "Erro",
                "Mensagem": "Cavalo não encontrado"
        }
    hStart = h(env, start[0], start[1])
    frontier = [(hStart, start, [start])]
    AccCost = {start: 0}
    expNodes = 0
    while frontier:
        f, (x, y), path = heapq.heappop(frontier)
        expNodes += 1
        if env.escape(x, y):
            return {
                "Resultado": "Fuga", 
                "Caminho": path, 
                "Nos_Expandidos": expNodes,
                "Custo": len(path) - 1
            }
        for nx, ny in env.possMovements(x, y):
            nAccCost = len(path)
            if (nx, ny) not in AccCost or nAccCost < AccCost[(nx, ny)]:
                AccCost[(nx, ny)] = nAccCost
                f_n = nAccCost + h(env, nx, ny)
                heapq.heappush(frontier, (f_n, (nx, ny), path + [(nx, ny)]))
    return {
        "Resultado": "Cercado",
        "Visitados": set(AccCost.keys()),
        "Nos_Expandidos": expNodes
    }

##### Heurística escolhida

In [5]:
def h(self, x, y):
    left_dist = x
    right_dist = (self.W - 1) - x
    top_dist = y
    bottom_dist = (self.H - 1) - y
    return min(left_dist, right_dist, top_dist, bottom_dist)

### 4. Resultados

...

In [6]:
import time

def executar_algoritmos(arch):
    print("\n" + "="*40)
    print(f" TESTANDO MAPA: {arch}")
    print("="*40)
    try:
        env = EncloseHorse(arch)
        algorithms = [
            ("Busca em Largura (BFS)", busca_largura),
            ("Busca em Profundidade (DFS)", busca_profundidade),
            ("Busca A*", busca_a_estrela)
        ]
        for name, search_func in algorithms:
            print(f"\n Executando {name}...")
            timeStart = time.time()
            res = search_func(env)
            timeEnd = time.time()
            timeMS = (timeEnd - timeStart) * 1000
            textResult = res.get('Resultado') or res.get('resultado')
            expandedNodes = res.get('Nos_Expandidos') or res.get('nos_expandidos')
            cost = res.get('Custo') or res.get('custo')
            visited = res.get('Visitados') or res.get('visitados')
            print(f"  - Resultado: {textResult}")
            print(f"  - Nós Expandidos: {expandedNodes}")
            print(f"  - Tempo: {timeMS:.2f} ms")
            if textResult == "Fuga":
                print(f"  - Custo do Caminho: {cost} passos")
            else:
                if hasattr(env, 'finalScore'):
                    points = env.finalScore(visited)
                    print(f"  - Pontuação da área cercada: {points} pontos")
        print("\n" + "-"*40)
        print(f" FIM DOS TESTES PARA: {arch}")
        print("-"*40)

    except FileNotFoundError:
        print(f" [ERRO]: O arquivo '{arch}' não foi encontrado.")
    except Exception as e:
        print(f" [ERRO INESPERADO] no arquivo {arch}: {e}")


In [ ]:
# executar_algoritmos("estados\\geometry.txt")
executar_algoritmos("estados\\entice.txt")


 TESTANDO MAPA: estados\entice.txt

 Executando Busca em Largura (BFS)...
  - Resultado: Fuga
  - Nós Expandidos: 76
  - Tempo: 1.00 ms
  - Custo do Caminho: 8 passos

 Executando Busca em Profundidade (DFS)...
  - Resultado: Fuga
  - Nós Expandidos: 9
  - Tempo: 0.00 ms
  - Custo do Caminho: 8 passos

 Executando Busca A*...
  - Resultado: Fuga
  - Nós Expandidos: 12
  - Tempo: 0.00 ms
  - Custo do Caminho: 8 passos

----------------------------------------
 FIM DOS TESTES PARA: estados\entice.txt
----------------------------------------


...

### 5. Discussão

...